# Chapter 18 — Advanced Time Series: ARIMA, VAR, Granger Causality
## University of Neuchâtel — Python for Econometrics

Topics:
1. AR(1) model — manual Yule-Walker vs ARIMA
2. ACF and PACF — manual vs statsmodels
3. ARIMA model fitting and forecasting
4. VAR model for multiple time series
5. Granger causality test

**Format:** One cell per step. Manual first, statsmodels second, `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import acf, adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Load Dataset: `phillips`

In [ ]:
df = woo.dataWoo('phillips')
print('Shape:', df.shape)
print(df[['year','inf','unem']].head(10))

In [ ]:
fig, axes = plt.subplots(2,1, figsize=(10,6))
axes[0].plot(df['year'], df['inf'])
axes[0].set_title('Inflation')
axes[0].set_ylabel('%')
axes[1].plot(df['year'], df['unem'], color='orange')
axes[1].set_title('Unemployment')
axes[1].set_ylabel('%')
axes[1].set_xlabel('Year')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch18_01_series.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2 — AR(1) Model

**Model:** $y_t = \mu + \rho y_{t-1} + e_t$

**Yule-Walker estimate:** $\hat{\rho} = Corr(y_t, y_{t-1})$

**When to use:** Series with autocorrelation — current value depends on past values.

### Step 2.1 — AR(1) coefficient via Yule-Walker (manual)

In [ ]:
y = df['inf'].values
rho_yw = np.corrcoef(y[1:], y[:-1])[0,1]
mu_yw  = y.mean() * (1 - rho_yw)
print(f'Yule-Walker rho: {rho_yw:.4f}')
print(f'Yule-Walker mu:  {mu_yw:.4f}')

### Step 2.2 — AR(1) via OLS of y_t on y_{t-1}

In [ ]:
df_ar = pd.DataFrame({'y': y[1:], 'y_lag': y[:-1]})
res_ar1_ols = smf.ols('y ~ y_lag', data=df_ar).fit()
print('AR(1) via OLS:')
print(f'  rho (y_lag): {res_ar1_ols.params["y_lag"]:.4f}')
print(f'  const:       {res_ar1_ols.params["Intercept"]:.4f}')

### Step 2.3 — AR(1) via ARIMA(1,0,0)

In [ ]:
arima_10 = ARIMA(y, order=(1,0,0)).fit()
print('ARIMA(1,0,0) parameters:')
print(arima_10.summary())
rho_arima = arima_10.params['ar.L1']
print(f'\nAR(1) coefficient: {rho_arima:.4f}')

### Step 2.4 — Compare Yule-Walker vs OLS vs ARIMA

In [ ]:
rho_ols = res_ar1_ols.params['y_lag']
print(f'Yule-Walker: {rho_yw:.4f}')
print(f'OLS:         {rho_ols:.4f}')
print(f'ARIMA:       {rho_arima:.4f}')
print(f'OLS vs YW match: {np.isclose(rho_ols, rho_yw, rtol=0.01)}')

## 3 — ACF and PACF

### Step 3.1 — Manual ACF (first 3 lags)

In [ ]:
acf1_man = np.corrcoef(y[1:], y[:-1])[0,1]
acf2_man = np.corrcoef(y[2:], y[:-2])[0,1]
acf3_man = np.corrcoef(y[3:], y[:-3])[0,1]
print(f'Manual ACF(1): {acf1_man:.4f}')
print(f'Manual ACF(2): {acf2_man:.4f}')
print(f'Manual ACF(3): {acf3_man:.4f}')

### Step 3.2 — ACF via statsmodels

In [ ]:
acf_sm = acf(y, nlags=3, fft=False)
print('statsmodels ACF(1,2,3):', acf_sm[1:4].round(4))
print(f'ACF(1) match: {np.isclose(acf1_man, acf_sm[1])}')
print(f'ACF(2) match: {np.isclose(acf2_man, acf_sm[2])}')

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4))
plot_acf(y, lags=10, ax=axes[0])
axes[0].set_title('ACF of Inflation')
plot_pacf(y, lags=10, ax=axes[1])
axes[1].set_title('PACF of Inflation')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch18_02_acf_pacf.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4 — ARIMA Model Selection and Forecasting

In [ ]:
# Check stationarity first
adf = adfuller(y, maxlag=1, autolag=None)
print(f'ADF test on inf: stat={adf[0]:.4f}, p={adf[1]:.4f}')
if adf[1] < 0.1:
    print('  Series appears stationary (I(0)) — use d=0')
else:
    print('  Series may be nonstationary — consider d=1')

In [ ]:
# Fit ARIMA(1,0,1)
arima_101 = ARIMA(y, order=(1,0,1)).fit()
print('ARIMA(1,0,1):')
print(arima_101.summary())

In [ ]:
# Compare AIC across models
models_aic = {}
for p in range(3):
    for q in range(3):
        try:
            m = ARIMA(y, order=(p,0,q)).fit()
            models_aic[f'ARIMA({p},0,{q})'] = m.aic
        except:
            pass
aic_df = pd.Series(models_aic).sort_values()
print('AIC by model (lower is better):')
print(aic_df.round(2))

In [ ]:
# Forecast 5 steps ahead
best_model_name = aic_df.index[0]
print(f'Best model by AIC: {best_model_name}')
p_best = int(best_model_name[6])
q_best = int(best_model_name[10])
arima_best = ARIMA(y, order=(p_best,0,q_best)).fit()
forecast = arima_best.forecast(steps=5)
print('5-step-ahead forecast:')
for i, f in enumerate(forecast):
    print(f'  t+{i+1}: {f:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(range(len(y)), y, label='Actual', linewidth=1.5)
ax.plot(range(len(y)), arima_best.fittedvalues, label='Fitted', linewidth=1.5, alpha=0.7)
forecast_idx = range(len(y), len(y)+5)
ax.plot(list(forecast_idx), forecast, 'r--o', label='Forecast', linewidth=2)
ax.set_xlabel('Time')
ax.set_ylabel('Inflation')
ax.set_title(f'{best_model_name} Fitted + Forecast')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch18_03_arima_forecast.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5 — VAR Model

**Vector Autoregression:** model two or more time series jointly.

$$\begin{pmatrix} inf_t \\ unem_t \end{pmatrix} = A_0 + A_1 \begin{pmatrix} inf_{t-1} \\ unem_{t-1} \end{pmatrix} + A_2 \begin{pmatrix} inf_{t-2} \\ unem_{t-2} \end{pmatrix} + \epsilon_t$$

**Why VAR?** Captures dynamic interactions between variables — each variable depends on its own lags and lags of all other variables.

### Step 5.1 — Check stationarity of both series

In [ ]:
for var in ['inf','unem']:
    adf_v = adfuller(df[var].dropna(), maxlag=1, autolag=None)
    print(f'ADF({var}): stat={adf_v[0]:.4f}, p={adf_v[1]:.4f}')

### Step 5.2 — Fit VAR model

In [ ]:
df_var = df[['inf','unem']].dropna()
var_model = VAR(df_var)
res_var = var_model.fit(maxlags=2, ic='aic')
print(res_var.summary())

### Step 5.3 — Granger Causality Test

**Granger causality:** Variable $x$ Granger-causes $y$ if past values of $x$ help
predict $y$ beyond what past values of $y$ alone can predict.

$H_0$: $x$ does NOT Granger-cause $y$

In [ ]:
# Does inflation Granger-cause unemployment?
gc_test = res_var.test_causality('unem', ['inf'], kind='f')
print('Granger causality test: inf → unem')
print(f'  F-stat: {gc_test.test_statistic:.4f}')
print(f'  p-value: {gc_test.pvalue:.4f}')
if gc_test.pvalue < 0.05:
    print('  Reject H0: inf Granger-causes unem')
else:
    print('  Fail to reject H0: inf does NOT Granger-cause unem')

In [ ]:
# Does unemployment Granger-cause inflation?
gc_test2 = res_var.test_causality('inf', ['unem'], kind='f')
print('Granger causality test: unem → inf')
print(f'  F-stat: {gc_test2.test_statistic:.4f}')
print(f'  p-value: {gc_test2.pvalue:.4f}')
if gc_test2.pvalue < 0.05:
    print('  Reject H0: unem Granger-causes inf')

In [ ]:
# Impulse Response Functions
try:
    irf = res_var.irf(10)
    fig = irf.plot(orth=False)
    plt.savefig(os.path.join(output_dir,'fig_ch18_04_irf.png'), dpi=150, bbox_inches='tight')
    plt.show()
except Exception as ex:
    print(f'IRF plot skipped: {ex}')

## 6 — Summary

In [ ]:
sumtab = pd.DataFrame({
    'Model': ['AR(1) Yule-Walker','AR(1) OLS','ARIMA best'],
    'rho_or_AR1': [rho_yw, rho_ols, rho_arima],
    'AIC': [np.nan, np.nan, arima_best.aic]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch18_ts_models.csv'), index=False)
print('Saved table_ch18_ts_models.csv')